# Tensile Test (PMDCo): from data entry to RDF

This notebook shows how to describe a uniaxial tensile test and convert it
to an RDF graph using the
[PMDCo measurement pattern](https://github.com/materialdigital/core-ontology/tree/main/patterns/measurement)
from the [Platform MaterialDigital Core Ontology (PMDCo)](https://w3id.org/pmd/co/)
— **without** TTO-specific property classes.

Results are identified by a free-text name label (`"Yield Strength"`,
`"Tensile Strength"`, …) rather than a fixed vocabulary. This makes the
schema flexible: any measured quantity can be recorded without extending
an ontology.

Use [`../TTO/`](../TTO/1_tensile_test_workflow.ipynb) instead when you need
TTO class annotations on individual mechanical properties.

## Graph structure

```
TensileTestingProcess  (pmdco:PMD_0000974)                  ← subClassOf OBI:assay
  realizes              (BFO_0000055) ─────────────────────► EvaluantRole (OBI_0000067)
  has_specified_input   (OBI_0000293) ─────────────────────► Specimen IRI (BFO_0000040)
    ↳ (on specimen) has_role (RO_0000087) ─────────────────► same EvaluantRole
    ↳ (on specimen) has_quality (RO_0000086) ──────────────► Quality (BFO_0000019)
  has_process_attribute (PMD_0000009) ─────────────────────► ParameterSpecification (PMD_0000013)
  has_specified_output  (OBI_0000299) ─────────────────────► MeasurementDatum (IAO_0000109)
    rdfs:label  ────────────────────────────────────────────── "Yield Strength" (free text)
    is_about              (IAO_0000136) ────────────────────► Specimen IRI
    is_quality_measurement_of (IAO_0000221) ────────────────► Quality
    has_value_specification (OBI_0001938) ──────────────────► ScalarValueSpec (OBI_0001931)
      has_specified_numeric_value (OBI_0001937) ─────────────── 310.0
      has_measurement_unit_label  (IAO_0000039) ─────────────── unit:MegaPascal
      specifies_value_of (OBI_0001927) ──────────────────────► Quality
      is_about           (IAO_0000136) ──────────────────────► Specimen IRI
```

## Environment setup

```bash
git clone https://github.com/Semantic-Dataspace/semantic-schemas.git
cd semantic-schemas
python3 -m venv .venv && source .venv/bin/activate
pip install semantic-schemas jupyterlab
jupyter lab
```

In [1]:
%pip install -q semantic-schemas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json, pathlib, rdflib
from semantic_schemas import Schema

HERE   = pathlib.Path().resolve()   # docs/
SCHEMA = HERE.parent                # tensile-test/PMDCo/

schema = Schema(SCHEMA)

print("Schema :", "/".join(SCHEMA.parts[-3:]))

Schema : step/tensile-test/PMDCo


## Step 1: Describe your tensile test

Edit `docs/example.input.json` with your data, then run this cell to load it.

| Field | Required | Description |
|---|---|---|
| `test_name` | yes | A name for this test |
| `specimen_iri` | yes | IRI of the specimen tested |
| `test_standard` | no | Standard followed, e.g. `"ISO 6892-1"` |
| `strain_rate` / `strain_rate_unit` | no | Strain rate and unit |
| `temperature` | no | Test temperature in °C |
| `gauge_length` / `gauge_length_unit` | no | Extensometer gauge length |
| `results` | no | `name`, `value`, `unit` — any property name is accepted |

Supported units: `"MPa"`, `"GPa"` for stress; `"%"` for elongation.

In [3]:
simplified = json.loads((HERE / "example.input.json").read_text())

print(json.dumps(simplified, indent=2))

{
  "test_name": "Tensile test 316L bar 1",
  "specimen_iri": "https://example.org/specimens/316L-tensile-bar-1",
  "test_standard": "ISO 6892-1",
  "strain_rate": 0.00025,
  "strain_rate_unit": "1/s",
  "temperature": 23,
  "results": [
    {
      "name": "Yield Strength",
      "value": 310,
      "unit": "MPa"
    },
    {
      "name": "Tensile Strength",
      "value": 620,
      "unit": "MPa"
    },
    {
      "name": "Elongation After Fracture",
      "value": 40,
      "unit": "%"
    },
    {
      "name": "Reduction of Area",
      "value": 65,
      "unit": "%"
    }
  ]
}


## Step 2: Convert to OO-LD

The transform converts your plain input into an OO-LD document.
Each result entry becomes a **MeasurementDatum** node (`obo:IAO_0000109`)
labelled with the property name and containing an embedded
**ScalarValueSpecification** (`obo:OBI_0001931`) with the numeric value and unit.

In [4]:
oold_doc = schema.transform(simplified)

print(json.dumps(oold_doc, indent=2))

{
  "conforms_to": "https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/characterization/step/tensile-test/PMDCo/#v1.0.0",
  "type": "pmdco:PMD_0000974",
  "id": "tensile-test-tensile-test-316l-bar-1",
  "label": "Tensile test 316L bar 1",
  "realizes": [
    {
      "type": "obo:OBI_0000067",
      "id": "tensile-test-tensile-test-316l-bar-1-evaluant-role",
      "realized_in": "tensile-test-tensile-test-316l-bar-1"
    }
  ],
  "has_specified_input": [
    "https://example.org/specimens/316L-tensile-bar-1"
  ],
  "has_process_attribute": [
    {
      "type": "pmdco:PMD_0000013",
      "id": "tensile-test-tensile-test-316l-bar-1-condition-standard",
      "parameter_label": "Test Standard",
      "parameter_unit": "ISO 6892-1"
    },
    {
      "type": "pmdco:PMD_0000013",
      "id": "tensile-test-tensile-test-316l-bar-1-condition-strain-rate",
      "parameter_label": "Strain Rate",
      "parameter_value": 0.00025,
      "parameter_unit": "1/s"
    },
    {
 

## Step 3: Convert to RDF

The OO-LD document is parsed into an RDF graph using the context from
`specs/schema.oold.yaml`:

| JSON field | Ontology IRI |
|---|---|
| `type` (process) | `rdf:type` → `pmdco:PMD_0000974` |
| `realizes` | `BFO_0000055` → evaluant role node (`OBI_0000067`) |
| `has_specified_input` | `OBI_0000293` → specimen (`BFO_0000040`) |
| `has_process_attribute` | `pmdco:PMD_0000009` |
| `measured_properties` | `OBI_0000299` (has_specified_output) → datum nodes |
| `label` (on datum) | `rdfs:label` — the property name |
| `is_about` (on datum) | `IAO_0000136` → specimen |
| `is_quality_measurement_of` | `IAO_0000221` → quality (`BFO_0000019`) |
| `has_value_spec` | `OBI_0001938` (has_value_specification) |
| `result_value` | `OBI_0001937` (has_specified_numeric_value) |
| `result_unit` | `IAO_0000039` (has_measurement_unit_label) |
| `specifies_value_of` | `OBI_0001927` → quality |
| `is_about` (on value spec) | `IAO_0000136` → specimen |

> **Tip:** Pass your own namespace as `base` so the generated IRIs use your
> data namespace. The cell below uses `"https://example.org/"` as a placeholder.

In [5]:
BASE_IRI = "https://example.org/"  # replace with your namespace in production

flat = schema.to_graph(simplified, base=BASE_IRI)

print(f'Graph contains {len(flat)} triples.\n')
print(flat.serialize(format='turtle'))

Graph contains 65 triples.

@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix obo: <http://purl.obolibrary.org/obo/> .
@prefix pmdco: <https://w3id.org/pmd/co/> .
@prefix qudt: <http://qudt.org/schema/qudt/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix uqudt: <http://qudt.org/vocab/unit/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<https://example.org/tensile-test-tensile-test-316l-bar-1> a pmdco:PMD_0000974 ;
    rdfs:label "Tensile test 316L bar 1" ;
    obo:BFO_0000055 <https://example.org/tensile-test-tensile-test-316l-bar-1-evaluant-role> ;
    obo:OBI_0000293 <https://example.org/specimens/316L-tensile-bar-1> ;
    obo:OBI_0000299 <https://example.org/tensile-test-tensile-test-316l-bar-1-result-elongation-after-fracture-smd>,
        <https://example.org/tensile-test-tensile-test-316l-bar-1-result-reduction-of-area-smd>,
        <https://example.org/tensile-test-tensile-test-316l-bar-1-result-tensile-strength-smd>,
        <https://example.org

In [6]:
out_ttl = HERE / "output_tensile_test.ttl"
out_ttl.write_text(flat.serialize(format="turtle"))
print(f"Written to {out_ttl.name}")

Written to output_tensile_test.ttl


## Step 4: Validate against SHACL shapes

The shape file `specs/shape.ttl` validates:
- The process node: must have `dcterms:conformsTo` and at least one specimen IRI.
- Each MeasurementDatum: must have an `rdfs:label` and a ScalarValueSpecification.
- Each ScalarValueSpecification: must have a positive numeric value and a supported unit.

In [7]:
conforms, violations = schema.validate(flat)

print(f"Conforms: {conforms}")
for v in violations:
    print(f"  Violation: {v}")

Conforms: True


## Step 5: Inspect the results

The SPARQL query below extracts the measured properties from the graph.
Because results are identified by `rdfs:label` rather than TTO class IRIs,
the query uses `rdfs:label` to retrieve the property name.

In [8]:
PMDCO = rdflib.Namespace("https://w3id.org/pmd/co/")

proc_iri = next(flat.subjects(rdflib.RDF.type, PMDCO["PMD_0000974"]))
print(f"Test IRI : {proc_iri}")
print(f"Label    : {flat.value(proc_iri, rdflib.RDFS.label)}")
print()

SPARQL = """
PREFIX obi:   <http://purl.obolibrary.org/obo/OBI_>
PREFIX iao:   <http://purl.obolibrary.org/obo/IAO_>
PREFIX pmdco: <https://w3id.org/pmd/co/>
PREFIX rdfs:  <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?name ?value ?unit
WHERE {
  ?test  a             pmdco:PMD_0000974 ;
         obi:0000299   ?datum .
  ?datum a             iao:0000109 ;
         rdfs:label    ?name ;
         obi:0001938   ?svs .
  ?svs   obi:0001937   ?value ;
         iao:0000039   ?unit .
}
ORDER BY ?name
"""

rows = list(flat.query(SPARQL))
if rows:
    print(f"{'Property name':<35}  {'Value':>8}  Unit")
    print("-" * 65)
    for r in rows:
        unit = str(r.unit).rsplit("/", 1)[-1]
        print(f"{str(r.name):<35}  {float(r.value):>8.3f}  {unit}")
else:
    print("No measured properties found in the graph.")

Test IRI : https://example.org/tensile-test-tensile-test-316l-bar-1
Label    : Tensile test 316L bar 1



Property name                           Value  Unit
-----------------------------------------------------------------
Elongation After Fracture              40.000  PERCENT
Reduction of Area                      65.000  PERCENT
Tensile Strength                      620.000  MegaPascal
Yield Strength                        310.000  MegaPascal


## Summary

| Step | What happens |
|---|---|
| 1 | Fill in `example.input.json` with the test name, specimen IRI, conditions, and results |
| 2 | The data is converted to an OO-LD document (ontology-mapped JSON) |
| 3 | The OO-LD is parsed into an RDF graph |
| 4 | The graph is validated against the SHACL shape |
| 5 | The measured properties are queried from the graph to confirm correctness |

---

## What comes next

This notebook used a hand-crafted JSON file as input. In practice, your measurement
data lives in an instrument export file.

[Notebook 2: from machine file to RDF](2_tensile_test_pmdco_csv_workflow.ipynb)
shows how to process a real Zwick/Roell export automatically using
`semantic-transformers`.

---

## TTO vs PMDCo

| | TTO schema | PMDCo schema (this notebook) |
|---|---|---|
| Process class | `pmdco:PMD_0000974` | `pmdco:PMD_0000974` |
| Result node type | `obi:OBI_0001931` (SVS) | `obo:IAO_0000109` (datum) + `obi:OBI_0001931` (SVS) |
| Property identification | TTO class IRI (e.g. `tto:YieldStrength`) | `rdfs:label` (free text) |
| SPARQL query | Must know TTO class IRIs | Uses `rdfs:label` — no ontology knowledge needed |